# Day 1 数据理解：AI4I 2020 工业设备故障数据集

## 项目目标

本项目基于 AI4I 2020 工业数据集，对设备故障影响因素进行分析，并在后续阶段建立基础预测模型，为预测性维护提供参考。

## 本阶段边界

今天只做数据理解：读取数据、认识字段、明确业务问题和后续分析方向。今天不做建模、不调参，也不进入复杂 EDA。

## 需要回答的 5 个业务问题

1. 整体设备故障率是多少？也就是全部记录中，`Machine failure = 1` 的比例有多高？
2. 不同产品类型的故障率是否有差异？例如 `L`、`M`、`H` 三类产品是否有不同的故障风险？
3. 刀具磨损越高，设备发生故障的可能性是否越高？这里重点关注 `Tool wear [min]` 和 `Machine failure` 的关系。
4. 温度、转速、扭矩和故障是否存在明显关系？也就是观察 `Air temperature [K]`、`Process temperature [K]`、`Rotational speed [rpm]`、`Torque [Nm]` 与故障之间的联系。
5. 各类故障原因的分布如何？例如 `TWF`、`HDF`、`PWF`、`OSF`、`RNF` 哪一种更常见？

## 1. 读取数据

下面先用最基础的 pandas 代码读取 CSV 文件。请注意：这里假设 notebook 是从项目根目录运行的，所以数据路径写成 `data/raw/ai4i2020.csv`。

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/raw/ai4i2020.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

## 2. 代码逐段解释

### `import pandas as pd`

这行代码的作用是导入 pandas 工具包，并把它简写成 `pd`。pandas 是 Python 里最常用的数据表处理工具，可以把 CSV 文件读成类似 Excel 表格的数据结构。

在这个项目里，AI4I 2020 数据集是一个 CSV 表格。我们用 pandas 来读取、查看和分析这张表。

你需要重点理解：这一步只是准备工具，还没有真正读取数据。

### `df = pd.read_csv("data/raw/ai4i2020.csv")`

这行代码会从 `data/raw/ai4i2020.csv` 读取原始数据，并保存到变量 `df` 中。

`df` 是 DataFrame 的常见缩写。你可以把它理解成 Python 里的“数据表”。每一行代表一条设备运行记录，每一列代表一个字段，例如温度、转速、扭矩、刀具磨损、是否故障等。

执行后重点观察：如果没有报错，说明文件路径正确，CSV 成功读进来了。如果报错 `FileNotFoundError`，通常说明当前 notebook 的运行目录不在项目根目录，或者路径写错了。

### `df.head()`

这行代码会显示数据表的前 5 行。

它的作用类似于先“扫一眼表格长什么样”。在这个项目里，你应该重点看：有哪些列、每列大概是什么格式、`Machine failure` 是否是 0/1、`TWF/HDF/PWF/OSF/RNF` 是否也是 0/1。

执行后重点观察：

- `UDI` 看起来像记录编号。
- `Product ID` 是产品编号。
- `Type` 是产品类型，通常有 `L`、`M`、`H`。
- 温度、转速、扭矩、刀具磨损是设备运行状态。
- `Machine failure` 是总故障标记。
- `TWF/HDF/PWF/OSF/RNF` 是不同故障原因的标记。

### `df.shape`

这行代码会返回数据表的形状，格式是 `(行数, 列数)`。

在这个项目里，它可以帮助你回答：“这个数据集有多少条记录？有多少个字段？”

执行后重点观察：行数代表样本数量，列数代表字段数量。AI4I 2020 数据集通常是 10000 行、14 列。

### `df.columns`

这行代码会显示所有列名。

在这个项目里，它的作用是帮你确认字段是否完整，以及字段名有没有空格、单位或特殊符号。例如 `Air temperature [K]` 里面带有单位 `[K]`，表示开尔文温度；`Tool wear [min]` 里面带有单位 `[min]`，表示分钟。

执行后重点观察：后续写代码时，列名必须和这里显示的完全一致，包括大小写、空格和中括号。

### `df.info()`

这行代码会显示数据表的整体信息，包括每列的非空数量和数据类型。

在这个项目里，它可以帮助你快速判断：有没有缺失值、哪些列是数字、哪些列是文本。

执行后重点观察：

- 每列的 `Non-Null Count` 是否等于总行数。如果相等，说明这一列没有缺失值。
- `Type` 和 `Product ID` 通常是文本类型。
- 温度、转速、扭矩、刀具磨损、故障标记通常是数值类型。
- `Machine failure` 是今天最重要的目标字段。

## 3. 字段说明表

| 字段名 | 中文含义 | 你理解的业务意义 | 类型 |
|---|---|---|---|
| `UDI` | 唯一记录编号 | 每一条设备运行记录的编号，主要用于区分样本，不直接代表设备状态好坏。 | 编号字段，整数 |
| `Product ID` | 产品编号 | 某个产品或工件的唯一标识，可以帮助追踪具体产品，但通常不直接作为判断故障规律的核心特征。 | 类别/文本字段 |
| `Type` | 产品类型 | 产品质量或类型等级，常见为 `L`、`M`、`H`。不同类型产品可能使用场景、负载或加工要求不同，因此故障率可能不同。 | 类别字段 |
| `Air temperature [K]` | 空气温度，单位 K | 设备周围环境温度。环境温度过高可能影响散热，从而增加设备异常风险。 | 数值字段，连续变量 |
| `Process temperature [K]` | 工艺温度，单位 K | 设备加工过程中的温度。它更接近设备实际运行状态，过高或异常变化可能和热相关故障有关。 | 数值字段，连续变量 |
| `Rotational speed [rpm]` | 转速，单位 rpm | 设备旋转部件每分钟转动次数。转速过高或过低都可能影响加工稳定性，也可能和功率、扭矩共同反映负载状态。 | 数值字段，连续变量 |
| `Torque [Nm]` | 扭矩，单位 Nm | 设备旋转时承受的力矩。扭矩可以理解成设备“用力程度”，扭矩过高可能说明负载较大，可能增加故障风险。 | 数值字段，连续变量 |
| `Tool wear [min]` | 刀具磨损时间，单位分钟 | 刀具已经使用或磨损的时间。磨损越久，刀具状态可能越差，发生刀具磨损故障或加工异常的风险可能越高。 | 数值字段，连续变量 |
| `Machine failure` | 是否发生设备故障 | 总故障标记。`1` 表示这条记录发生了设备故障，`0` 表示没有故障。它是后续预测性维护建模最核心的目标字段。 | 目标字段，二分类 0/1 |
| `TWF` | 刀具磨损故障 | Tool Wear Failure。表示是否因为刀具磨损导致故障。它是具体故障原因之一，不是总故障率本身。 | 故障原因标记，0/1 |
| `HDF` | 散热故障 | Heat Dissipation Failure。表示是否因为散热或热量无法有效释放导致故障，通常和温度状态有关。 | 故障原因标记，0/1 |
| `PWF` | 功率故障 | Power Failure。表示是否因为功率状态异常导致故障，可能和转速、扭矩共同反映的设备功率负载有关。 | 故障原因标记，0/1 |
| `OSF` | 过载故障 | Overstrain Failure。表示是否因为设备承受过大应力或过载导致故障，可能和扭矩、刀具磨损、产品类型有关。 | 故障原因标记，0/1 |
| `RNF` | 随机故障 | Random Failure。表示随机性故障，可能不容易由当前这些传感器字段解释。分析时要注意它可能没有明显规律。 | 故障原因标记，0/1 |

## 4. 容易混淆的字段提醒

- `Machine failure` 是总目标字段，表示是否发生故障；`TWF/HDF/PWF/OSF/RNF` 是具体故障类型或原因标记。
- `Air temperature [K]` 是环境空气温度，`Process temperature [K]` 是加工过程温度，两者不是同一个概念。
- `Rotational speed [rpm]` 是转速，`Torque [Nm]` 是扭矩。简单理解：转速看“转得多快”，扭矩看“用力多大”。
- `Product ID` 是产品编号，`Type` 是产品类型。编号通常更像身份标签，类型更适合用来比较不同类别的故障差异。
- 温度单位是 K，不是摄氏度。初学阶段不用急着换算，但要知道它不是日常说的摄氏温度。

## 5. 今天学完后应该能掌握的知识点

1. 知道如何用 `pd.read_csv()` 把 CSV 文件读成 DataFrame。
2. 知道 `df.head()` 用来快速查看前几行数据。
3. 知道 `df.shape` 用来查看数据规模，也就是行数和列数。
4. 知道 `df.columns` 用来查看所有字段名，后续写代码必须严格匹配字段名。
5. 知道 `df.info()` 用来查看字段类型和缺失情况。
6. 能区分目标字段、普通特征字段、故障原因字段。
7. 能用业务语言解释这个数据集为什么和预测性维护有关。

## 6. Day 1 自检标准

完成今天内容后，你应该可以不看网页，自己说清楚下面几件事：

1. 这个数据集是干什么的：它记录了工业设备在不同运行状态下是否发生故障，用来分析设备故障影响因素，并为后续预测性维护建模做准备。
2. 目标字段是哪一列：`Machine failure`，其中 `1` 表示发生故障，`0` 表示没有故障。
3. 主要特征有哪些：产品类型 `Type`，空气温度、工艺温度、转速、扭矩、刀具磨损，以及具体故障原因标记 `TWF/HDF/PWF/OSF/RNF`。
4. 后面准备分析什么：整体故障率、不同产品类型的故障率差异、刀具磨损和故障的关系、温度/转速/扭矩和故障的关系、不同故障类型的分布。
5. 今天不需要做什么：不建模、不调参、不追求复杂图表，先把字段和业务问题理解清楚。